## 0. Setup — Download Dataset dari Google Drive (Prioritas) atau Roboflow

Notebook ini memprioritaskan dataset dari **Google Drive** (`MyDrive/ultra-milk-yolo-ready.zip` atau folder `MyDrive/ultra-milk-yolo-ready`).

Kalau tidak ada di Drive, otomatis fallback ke download langsung dari Roboflow.

Link Roboflow: `https://app.roboflow.com/ds/qFlbnnIoI8?key=jjFHrlZqjO`


In [ ]:
# ============================================
# Langkah 1: Siapkan dataset (Google Drive -> Roboflow fallback)
# ============================================
from google.colab import drive
from google.colab import files
from pathlib import Path
import zipfile
import os
import shutil

# Mount Drive
drive.mount('/content/drive')

# Konfigurasi
ROBOFLOW_URL = "https://app.roboflow.com/ds/qFlbnnIoI8?key=jjFHrlZqjO"
DRIVE_ZIP = Path('/content/drive/MyDrive/ultra-milk-yolo-ready.zip')
DRIVE_DIR = Path('/content/drive/MyDrive/ultra-milk-yolo-ready')
CONTENT_PATH = Path('/content/ultra-milk-yolo-ready')
ZIP_PATH = Path('/content/roboflow.zip')
TEMP_EXTRACT = Path('/content/tmp_dataset_extract')

def ensure_dataset(base: Path) -> bool:
    return base.exists() and (base / 'data.yaml').exists()

def rename_classes(data_yaml_path: Path) -> None:
    text = data_yaml_path.read_text(encoding='utf-8')
    old = "names: ['um_normal', 'um_rusak']"
    new = "names: ['passed', 'damaged']"
    if old in text:
        text = text.replace(old, new)
        data_yaml_path.write_text(text, encoding='utf-8')
        print('Class names renamed: um_normal -> passed, um_rusak -> damaged')
    else:
        print('Class names already renamed or different format')

def fix_yaml_paths(data_yaml_path: Path, dataset_root: Path) -> None:
    text = data_yaml_path.read_text(encoding='utf-8')
    text = text.replace('train: ../train/images', 'train: train/images')
    text = text.replace('val: ../valid/images', 'val: valid/images')
    text = text.replace('test: ../test/images', 'test: test/images')
    text = text.replace('path: ultra-milk-yolo-ready', f'path: {dataset_root}')
    data_yaml_path.write_text(text, encoding='utf-8')
    print(f'Patched data.yaml paths for root: {dataset_root}')

def extract_zip_to(zip_path: Path, target_dir: Path) -> Path | None:
    if target_dir.exists():
        shutil.rmtree(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(target_dir)
    zip_path.unlink()
    if ensure_dataset(target_dir):
        return target_dir
    for subdir in target_dir.iterdir():
        if subdir.is_dir() and ensure_dataset(subdir):
            return subdir
    return None

base = None

# Prioritas 1: Cek folder dataset di Google Drive
if ensure_dataset(DRIVE_DIR):
    base = DRIVE_DIR
    print(f"Menggunakan dataset dari Drive: {base}")
elif DRIVE_ZIP.exists():
    print(f"Mengekstrak zip dari Drive...")
    base = extract_zip_to(DRIVE_ZIP, TEMP_EXTRACT)
    if base is not None:
        if base != CONTENT_PATH and base.parent == TEMP_EXTRACT:
            if CONTENT_PATH.exists():
                shutil.rmtree(CONTENT_PATH)
            base.replace(CONTENT_PATH)
            base = CONTENT_PATH
        elif base != CONTENT_PATH and base == TEMP_EXTRACT:
            if CONTENT_PATH.exists():
                shutil.rmtree(CONTENT_PATH)
            CONTENT_PATH.mkdir(parents=True, exist_ok=True)
            for item in TEMP_EXTRACT.iterdir():
                dest = CONTENT_PATH / item.name
                if dest.exists():
                    if dest.is_dir():
                        shutil.rmtree(dest)
                    else:
                        dest.unlink()
                shutil.move(str(item), str(dest))
            base = CONTENT_PATH
        print(f"Dataset dari Drive zip siap di: {base}")
    else:
        print('Gagal ekstrak zip Drive, mencoba Roboflow...')
else:
    print('Dataset tidak ditemukan di Drive, mendownload dari Roboflow...')

# Prioritas 2: Fallback download dari Roboflow
if base is None:
    os.system(f'curl -L "{ROBOFLOW_URL}" -o {ZIP_PATH}')
    if ZIP_PATH.exists() and ZIP_PATH.stat().st_size > 1000:
        base = extract_zip_to(ZIP_PATH, TEMP_EXTRACT)
        if base is not None:
            if base != CONTENT_PATH and base.parent == TEMP_EXTRACT:
                if CONTENT_PATH.exists():
                    shutil.rmtree(CONTENT_PATH)
                base.replace(CONTENT_PATH)
                base = CONTENT_PATH
            elif base != CONTENT_PATH and base == TEMP_EXTRACT:
                if CONTENT_PATH.exists():
                    shutil.rmtree(CONTENT_PATH)
                CONTENT_PATH.mkdir(parents=True, exist_ok=True)
                for item in TEMP_EXTRACT.iterdir():
                    dest = CONTENT_PATH / item.name
                    if dest.exists():
                        if dest.is_dir():
                            shutil.rmtree(dest)
                        else:
                            dest.unlink()
                    shutil.move(str(item), str(dest))
                base = CONTENT_PATH
            print(f"Dataset dari Roboflow siap di: {base}")
        else:
            raise FileNotFoundError('data.yaml tidak ditemukan setelah ekstrak Roboflow.')
    else:
        raise FileNotFoundError('Gagal download dataset dari Roboflow.')

fix_yaml_paths(base / 'data.yaml', base)
rename_classes(base / 'data.yaml')

# Set global paths
DATA_BASE = str(base)
DATA_YAML = str(base / 'data.yaml')

print('\nKonfigurasi dataset:')
print(f'  Base   : {DATA_BASE}')
print(f'  YAML   : {DATA_YAML}')


## 1. Install Ultralytics + Cek GPU

In [ ]:
%pip install -q ultralytics

import torch
print('PyTorch version    :', torch.__version__)
print('CUDA available     :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU                :', torch.cuda.get_device_name(0))


## 2. Train Model YOLOv8

In [ ]:
from ultralytics import YOLO

use_cuda = torch.cuda.is_available()

print('Data yaml  :', DATA_YAML)
print('Epochs     :', 10)
print('Device     :', 'cuda' if use_cuda else 'cpu')

model = YOLO('yolov8n.pt')
model.train(
    data=DATA_YAML,
    epochs=10,
    imgsz=640,
    batch=16,
    device='cuda' if use_cuda else 'cpu',
    workers=2,
    cache=False,
    project='/content/runs/detect',
    name='train',
    exist_ok=True,
)


## 3. Download best.pt

In [ ]:
from google.colab import files
from pathlib import Path

best = Path('/content/runs/detect/train/weights/best.pt')
if not best.exists():
    raise FileNotFoundError(f'best.pt tidak ditemukan di {best}. Pastikan training selesai.')

files.download(str(best))
print('Downloaded:', best)
